## Demonstration

---

### Cell 1: Imports

In [14]:
import torch
import torch.nn as nn

layer = nn.Linear(in_features=4, out_features=2)
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])  # shape (1,4)
y = layer(x)
print(y.shape)

torch.Size([1, 2])



###  **Cell 1: Imports**


In [15]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import re
import random
from collections import Counter


---

### Cell 2: Synthetic Dataset Creation

In [16]:

# Synthetic Movie Review Dataset
class SyntheticMovieDataset:
    def __init__(self):
        positive_reviews = [
            "This movie is absolutely fantastic and amazing",
            "Brilliant acting and wonderful storyline throughout",
            "Excellent film with great performances by all actors",
            "Outstanding cinematography and superb direction",
            "Incredible movie with beautiful scenes and perfect music",
            "Amazing plot with excellent character development",
            "Wonderful film that exceeded all my expectations",
            "Fantastic acting and brilliant screenplay writing",
            "Superb movie with outstanding visual effects",
            "Excellent story with amazing performances",
        ] * 10

        negative_reviews = [
            "This movie is terrible and completely boring",
            "Awful acting and horrible storyline throughout",
            "Poor film with bad performances by actors",
            "Terrible cinematography and awful direction",
            "Horrible movie with ugly scenes and terrible music",
            "Bad plot with poor character development",
            "Awful film that disappointed all expectations",
            "Terrible acting and poor screenplay writing",
            "Bad movie with horrible visual effects",
            "Poor story with terrible performances",
        ] * 10

        self.data = [(t, 1) for t in positive_reviews] + [(t, 0) for t in negative_reviews]
        random.shuffle(self.data)

    def get_train_data(self):
        return self.data[:160]

    def get_test_data(self):
        return self.data[160:]


---

### Cell 3: Tokenizer

In [17]:

# Tokenizer and Vocabulary Builder
class SimpleTokenizer:
    def __init__(self):
        self.word_to_idx = {"<pad>": 0, "<unk>": 1}
        self.idx_to_word = {0: "<pad>", 1: "<unk>"}
        self.vocab_size = 2

    def tokenize(self, text):
        text = re.sub(r'[^\w\s]', '', text.lower())
        return text.split()

    def build_vocab(self, texts):
        word_counts = Counter()
        for text in texts:
            tokens = self.tokenize(text)
            word_counts.update(tokens)
        for word, _ in word_counts.most_common(1000):
            if word not in self.word_to_idx:
                self.word_to_idx[word] = self.vocab_size
                self.idx_to_word[self.vocab_size] = word
                self.vocab_size += 1

    def text_to_indices(self, text, max_length=50):
        tokens = self.tokenize(text)[:max_length]
        indices = [self.word_to_idx.get(t, 1) for t in tokens]
        while len(indices) < max_length:
            indices.append(0)
        return indices


---

### 🧩 **Cell 4: Dataset Wrapper**



In [18]:

# PyTorch Dataset Wrapper
class MovieDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=50):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, label = self.data[idx]
        indices = self.tokenizer.text_to_indices(text, self.max_length)
        return torch.tensor(indices, dtype=torch.long), torch.tensor(label, dtype=torch.long)


---

### Cell 5: Transformer Classifier Model



In [19]:

# Transformer Model
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, num_layers=2, num_classes=2, max_seq_len=50):
        super().__init__()
        self.embed_dim = embed_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoding = self._create_positional_encoding(max_seq_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=128, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 32), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(32, num_classes)
        )

    def _create_positional_encoding(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)

    def forward(self, x):
        seq_len = x.size(1)
        mask = (x == 0)
        x = self.embedding(x) * np.sqrt(self.embed_dim)
        x = x + self.pos_encoding[:, :seq_len, :].to(x.device)
        x = self.transformer(x, src_key_padding_mask=mask)
        valid = (~mask).float().unsqueeze(-1)
        x = (x * valid).sum(dim=1) / valid.sum(dim=1)
        return self.classifier(x)


---

### Cell 6: Data Preparation



In [20]:

# Data Preparation
print("Creating synthetic dataset...")
dataset_creator = SyntheticMovieDataset()
train_data = dataset_creator.get_train_data()
test_data = dataset_creator.get_test_data()

tokenizer = SimpleTokenizer()
all_texts = [t for t, _ in train_data + test_data]
tokenizer.build_vocab(all_texts)
print(f"Vocabulary size: {tokenizer.vocab_size}")

train_dataset = MovieDataset(train_data, tokenizer)
test_dataset = MovieDataset(test_data, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


Creating synthetic dataset...
Vocabulary size: 52


---

### Cell 7: Training Loop

In [21]:

# Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerClassifier(vocab_size=tokenizer.vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 5
print("\nStarting training...")

for epoch in range(num_epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for texts, labels in train_loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {total_loss/len(train_loader):.4f} Accuracy: {100*correct/total:.2f}%")



Starting training...
Epoch [1/5] Loss: 0.6433 Accuracy: 70.62%
Epoch [1/5] Loss: 0.6433 Accuracy: 70.62%
Epoch [2/5] Loss: 0.4600 Accuracy: 94.38%
Epoch [2/5] Loss: 0.4600 Accuracy: 94.38%
Epoch [3/5] Loss: 0.1862 Accuracy: 100.00%
Epoch [3/5] Loss: 0.1862 Accuracy: 100.00%
Epoch [4/5] Loss: 0.0525 Accuracy: 100.00%
Epoch [4/5] Loss: 0.0525 Accuracy: 100.00%
Epoch [5/5] Loss: 0.0205 Accuracy: 100.00%
Epoch [5/5] Loss: 0.0205 Accuracy: 100.00%


---

### 🧩 **Cell 8: Testing / Evaluation**


In [22]:

# Testing
model.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for texts, labels in test_loader:
        texts, labels = texts.to(device), labels.to(device)
        outputs = model(texts)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
test_accuracy = 100 * test_correct / test_total
print(f"\nTest Accuracy: {test_accuracy:.2f}%")



Test Accuracy: 100.00%


---

### 🧩 **Cell 9: Interactive Predictions**



In [23]:
# Prediction Function
def predict_sentiment(text):
    model.eval()
    with torch.no_grad():
        indices = tokenizer.text_to_indices(text)
        tensor = torch.tensor(indices).unsqueeze(0).to(device)
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        pred = torch.argmax(output, dim=1).item()
        conf = probs[0][pred].item()
        return ("Positive" if pred == 1 else "Negative"), conf

# Try with custom reviews
test_reviews = [
    "This movie is absolutely fantastic and amazing!",
    "Terrible film with awful acting and poor story.",
    "Great performances and excellent cinematography.",
    "Boring movie with bad direction and terrible script."
]

print("\n=== SAMPLE REVIEW TESTS ===")
for r in test_reviews:
    s, c = predict_sentiment(r)
    print(f"Review: {r}\nPrediction: {s} (Confidence: {c:.3f})\n{'-'*50}")



=== SAMPLE REVIEW TESTS ===
Review: This movie is absolutely fantastic and amazing!
Prediction: Positive (Confidence: 0.997)
--------------------------------------------------
Review: Terrible film with awful acting and poor story.
Prediction: Negative (Confidence: 0.989)
--------------------------------------------------
Review: Great performances and excellent cinematography.
Prediction: Positive (Confidence: 0.997)
--------------------------------------------------
Review: Boring movie with bad direction and terrible script.
Prediction: Negative (Confidence: 0.984)
--------------------------------------------------


---

✅ **Result:**
You now have a **clean, step-by-step Transformer NLP project** organized for execution in VSCode or Jupyter Notebook.
Each cell logically handles one part of the workflow:
**data → tokenization → dataset → model → training → evaluation → inference.**